# Laya V2.9 — Can an Attention-Free MultiscaleCeNN Replace Full Attention?

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V29_MultiscaleCeNN_FullAttention_Colab.ipynb)

**Main test:** Replace only **ModernBERT full-attention layer 15**. All other layers are unchanged during the isolated experiment. The *replacement itself* is strictly attention-free (no Q–K dot products, no SDPA, no token-to-token softmax, no external attention teacher at inference). The frozen teacher retains native attention for distillation, as expected.

Candidates: original **MultiscaleCeNN 256×2**, large **MultiscaleCeNN 512×4**, **Adaptive Hierarchical MultiscaleCeNN 384×3**, and **Adaptive Hierarchical MultiscaleCeNN 512×4**. The adaptive architecture uses learnable per-token multiscale routing, contractive cellular recurrence, multiresolution pooled states, global mean/RMS context, and a pretrained *value-only* direct path. It is not a claim of a literature-first new mechanism.

**Data rigor:** case-disjoint train/probe/audit groups from the original training split. Candidate selection uses the probe subset only. Audit metrics are descriptive and do not select the winner. Official test data is not touched. 400 equal-budget successful optimizer updates per architecture, optional 1200-step refinement of probe-selected finalists. A longer window or more updates are *not* treated as proof of a better method without downstream agreement and latency.

**Success gates:** local cosine ≥ 0.95, NMSE ≤ 0.10, teacher-decision agreement ≥ 0.95, mean JS ≤ 0.02. These are research targets, not achieved results. Report measured attention-layer and whole-model latency; do not claim a speedup without a measured value.

**Optional follow-up:** separately train and insert attention-free CeNN modules for **every** full-attention layer and evaluate cumulative decision agreement, holding existing sliding layers unchanged. Disabled by default as it is computationally expensive.

**Limitation:** This is primarily a task- and model-specific replacement study, not proof of general long-context attention equivalence. The report includes actual valid-token length distributions so short inputs are not mistaken for long-context success.


In [ ]:
#@title 1. Install exact training dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2029
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 600 #@param {type:"integer"}
DEV_CASES = 200 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 48 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 14 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 18 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V29_MultiscaleCeNN")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert STEPS == 400 and DEV_CASES >= 4, "Use 400-step screening and a meaningful held-out dev split"
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
print("screen/refine:", STEPS, REFINE_STEPS, "official test used: NO")


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_embeddings=entry["pos"],
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V2.6 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


In [ ]:
#@title 4. Attention-free adaptive multiscale cellular encoder
class AdaptiveHierarchicalCeNN(nn.Module):
    """Attention-free token mixer.

    Uses per-scale masked pooling, finite-step contractive depthwise cellular
    recurrence, token-dependent scale gates, and global channelwise moments.
    Absolutely no Q-K score matrices, SDPA, linear attention, or cross-token
    content-based attention; no sequence-dependent learned token pair scores.
    """
    architecture = "adaptive_hierarchical_multiscale_cenn"

    def __init__(self, original, cell_dim=384, steps=3, scales=(1,4,16,64)):
        super().__init__()
        self.config = original.config
        self.cell_dim = int(cell_dim)
        self.steps = int(steps)
        self.scales = tuple(int(s) for s in scales)
        if self.cell_dim < 1 or self.steps < 1 or not self.scales or any(s < 1 for s in self.scales):
            raise ValueError("Invalid adaptive CeNN dimensions")
        d = int(self.config.hidden_size)
        self.down = nn.Linear(d, self.cell_dim, bias=False)
        self.up = nn.Linear(self.cell_dim, d, bias=False)
        self.norm = nn.LayerNorm(self.cell_dim)
        self.template = nn.Parameter(torch.zeros(self.cell_dim, 1, 5))
        self.leak_logit = nn.Parameter(torch.zeros(self.cell_dim))
        self.input_gain = nn.Parameter(torch.ones(self.cell_dim))
        self.global_proj = nn.Linear(2 * self.cell_dim, self.cell_dim)
        self.global_gain = nn.Parameter(torch.full((self.cell_dim,), 0.1))
        self.route = nn.Linear(self.cell_dim, len(self.scales))
        self.route_global = nn.Linear(self.cell_dim, len(self.scales), bias=False)
        self.route_temperature = nn.Parameter(torch.tensor(0.0))
        self.readout_gain = nn.Parameter(torch.tensor(0.0))
        self.direct_gate = nn.Parameter(torch.tensor(-1.0))
        self.out_drop = copy.deepcopy(original.out_drop)
        # Initialize cell read/write paths using pretrained VALUE+output maps.
        self.value_proj = nn.Linear(d, d, bias=original.Wqkv.bias is not None)
        self.value_out = copy.deepcopy(original.Wo)
        with torch.no_grad():
            self.value_proj.weight.copy_(original.Wqkv.weight[2*d:3*d])
            if self.value_proj.bias is not None:
                self.value_proj.bias.copy_(original.Wqkv.bias[2*d:3*d])
            pick = min(self.cell_dim, d)
            self.down.weight[:pick].copy_(self.value_proj.weight[:pick])
            self.up.weight[:, :pick].copy_(original.Wo.weight[:, :pick])
            nn.init.zeros_(self.template)
            self.template[:, 0, 2] = 0.05
            nn.init.zeros_(self.route.weight)
            nn.init.zeros_(self.route.bias)
            nn.init.zeros_(self.route_global.weight)
            nn.init.normal_(self.global_proj.weight, std=0.002)
            nn.init.zeros_(self.global_proj.bias)
        self.value_proj.requires_grad_(False)
        self.value_out.requires_grad_(False)

    def forward(self, hidden_states, attention_mask=None, **kwargs):
        # No RoPE needed: recurrence and multiscale pools encode locality/order.
        b, t, _ = hidden_states.shape
        if attention_mask is None:
            valid = torch.ones((b, t), device=hidden_states.device, dtype=torch.bool)
        elif attention_mask.ndim == 2:
            valid = attention_mask.bool()
        else:
            # Allow standard ModernBERT full-attention 4-D additive/bool masks.
            from tinycenn_lm.laya_lab.core import _valid_tokens
            valid = _valid_tokens(attention_mask, hidden_states)
        m = valid[:, :, None].to(hidden_states.dtype)
        u = self.norm(self.down(hidden_states)) * m
        counts = m.sum(1).clamp_min(1)
        gmean = u.sum(1) / counts
        grms = torch.sqrt((u.square() * m).sum(1) / counts + 1e-6)
        glob = self.global_proj(torch.cat([gmean, grms], dim=-1))
        context = (glob * self.global_gain[None, :])[:, :, None]
        kernels = torch.tanh(self.template.float())
        kernels = 0.90 * kernels / kernels.abs().sum(-1, keepdim=True).clamp_min(1.0)
        leak = torch.sigmoid(self.leak_logit.float())[None, :, None]
        branches = []
        for scale in self.scales:
            pad = (-t) % scale
            upad = F.pad(u.transpose(1,2), (0, pad))
            mpad = F.pad(m.transpose(1,2), (0, pad))
            n = (t + pad) // scale
            cnt = mpad.reshape(b, 1, n, scale).sum(-1).clamp_min(1)
            pooled = upad.reshape(b, self.cell_dim, n, scale).sum(-1) / cnt
            pmask = (mpad.reshape(b, 1, n, scale).sum(-1) > 0).to(u.dtype)
            drive = pooled * self.input_gain[None, :, None] + context
            state = torch.tanh(pooled) * pmask
            for _ in range(self.steps):
                feedback = F.conv1d(state, kernels.to(state.dtype), padding=2, groups=self.cell_dim)
                state = ((1 - leak) * state + leak * torch.tanh(feedback + drive)) * pmask
            branches.append(state.repeat_interleave(scale, dim=-1)[..., :t].transpose(1, 2))
        route_input = self.route(u) + self.route_global(glob)[:, None, :]
        routes = torch.softmax(route_input / self.route_temperature.clamp(-1,1).exp(), dim=-1)
        stack = torch.stack(branches, dim=2)  # B,T,scales,C
        mixed = (stack * routes[..., None]).sum(2) * m
        local = self.up(mixed) * self.readout_gain.clamp(-2,2).exp()
        direct = self.value_out(self.value_proj(hidden_states)) * m
        gate = self.direct_gate.clamp(-5,5).sigmoid()
        out = ((1-gate)*local + gate*direct) * m
        return self.out_drop(out), None

    def config_dict(self):
        return dict(architecture=self.architecture, cell_dim=self.cell_dim,
                    steps=self.steps, scales=list(self.scales),
                    attention_free=True, global_context="mean_rms",
                    token_routing="per_token_multiscale_gates",
                    value_only_pretrained_anchor=True)

CANDIDATE_INFO = {
    "cenn_v25": dict(label="Original MultiscaleCeNN 256x2", cell_dim=256,
                     steps=2, scales=(1,4,16), lr=(0.01,0.003)),
    "cenn_large": dict(label="Large MultiscaleCeNN 512x4", cell_dim=512,
                       steps=4, scales=(1,2,4,8,16,32), lr=(0.008,0.002)),
    "adaptive_384": dict(label="Adaptive Hierarchical CeNN 384x3", cell_dim=384,
                         steps=3, scales=(1,4,16,64), lr=(0.003,0.0006)),
    "adaptive_512": dict(label="Adaptive Hierarchical CeNN 512x4", cell_dim=512,
                         steps=4, scales=(1,2,8,32,128), lr=(0.003,0.0006)),
}
CANDIDATE_NAMES = tuple(CANDIDATE_INFO)

def build_candidate(name, original):
    recipe = CANDIDATE_INFO[name]
    if name.startswith("adaptive_"):
        model = AdaptiveHierarchicalCeNN(
            original, cell_dim=recipe["cell_dim"],
            steps=recipe["steps"], scales=recipe["scales"],
        )
    else:
        model = MultiscaleCeNN(
            original, cell_dim=recipe["cell_dim"],
            steps=recipe["steps"], scales=recipe["scales"],
        )
    return model.to(device)

def candidate_parameters(module):
    return [p for p in module.parameters() if p.requires_grad]

print("Candidates (ALL attention-free):")
for name, info in CANDIDATE_INFO.items():
    print(" ", name, "->", info["label"])


In [ ]:
#@title 5. Preflight: ModernBERT compatibility and finite nonzero gradients
sample_batch = batch(probe_items, 0, min(2, BATCH))
sample_io = attn_io(teacher, LAYER, sample_batch)
for name in CANDIDATE_NAMES:
    model = build_candidate(name, teacher.encoder.layers[LAYER].attn)
    model.train()
    model.out_drop.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        pred, _ = model(
            sample_io["x"], position_embeddings=sample_io["pos"],
            attention_mask=sample_io["mask"],
        )
        m = sample_batch["attention_mask"].bool()[:, :, None].float()
        loss = (((pred.float()-sample_io["y"].float())**2)*m).sum() / (
            m.sum()*pred.shape[-1]).clamp_min(1)
    require_finite(pred, name+" preflight output")
    require_finite(loss, name+" preflight loss")
    params = candidate_parameters(model)
    grads = torch.autograd.grad(loss, params, allow_unused=True)
    gsum = sum(float(g.abs().sum()) for g in grads if g is not None and torch.isfinite(g).all())
    if not np.isfinite(gsum) or gsum <= 0:
        raise RuntimeError(name+": no finite nonzero gradients")
    print(f"{name:<18} PASS | loss={float(loss):.5f} | trainable={sum(p.numel() for p in params):,}")
    del model, pred, loss, grads
    torch.cuda.empty_cache()
del sample_batch, sample_io


In [ ]:
#@title 6. Fixed 400-step local fit with automatic probe checkpoint selection
def state_cpu(module):
    return {key: value.detach().cpu().clone() for key, value in module.state_dict().items()}

def metric_score(metrics):
    return metrics["nmse"] + 1.10 * (1.0 - metrics["cosine"])

def cosine_lr(update, total, start, end):
    if total <= 1: return float(end)
    fraction = min(1., update / float(total - 1))
    return end + 0.5 * (start-end) * (1 + math.cos(math.pi*fraction))

def fit_local(name, model, train_cache, probe_cache, steps, phase="screen"):
    start, end = CANDIDATE_INFO[name]["lr"]
    if phase == "refine":
        start, end = start * .35, end * .25
    parameters = candidate_parameters(model)
    opt = torch.optim.AdamW(parameters, lr=start, betas=(0.9,0.95), weight_decay=2e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16, init_scale=128.)
    best_metrics = local_metrics_from_cache(model, probe_cache)
    best_state = state_cpu(model)
    best_step = 0
    best_score = metric_score(best_metrics)
    history = [{"step": 0, **best_metrics}]
    updates = attempts = 0
    print(f"{name} {phase} start: NMSE={best_metrics['nmse']:.4f} cos={best_metrics['cosine']:.4f}")
    while updates < steps:
        if attempts > steps + 300:
            raise RuntimeError(name+": too many AMP overflows")
        for pg in opt.param_groups:
            pg["lr"] = cosine_lr(updates, steps, start, end)
        entry = train_cache[attempts % len(train_cache)]
        model.train()
        model.out_drop.eval()  # deterministic frozen-teacher target
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            pred, _ = model(entry["x"], position_embeddings=entry["pos"],
                            attention_mask=entry["mask"])
            m = entry["valid"][:, :, None].float()
            target = entry["y"].float()
            diff = pred.float() - target
            mse = ((diff**2)*m).sum() / (m.sum()*pred.shape[-1]).clamp_min(1)
            power = ((target**2)*m).sum() / (m.sum()*pred.shape[-1]).clamp_min(1)
            nmse = mse / power.clamp_min(1e-8)
            cos = F.cosine_similarity(
                pred.float()[entry["valid"]], target[entry["valid"]], dim=-1,
            ).mean()
            loss = nmse + 1.10*(1-cos)
        require_finite(loss, name+" loss")
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(parameters, 1.0)
        old_scale = scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        attempts += 1
        if scaler.is_enabled() and scaler.get_scale() < old_scale:
            continue
        updates += 1
        if updates == 1 or updates % CHECK_EVERY == 0 or updates == steps:
            probe = local_metrics_from_cache(model, probe_cache)
            history.append({"step": updates, **probe})
            if metric_score(probe) < best_score:
                best_metrics, best_score = probe, metric_score(probe)
                best_step, best_state = updates, state_cpu(model)
            print(f"  {updates:04d}/{steps}: NMSE={probe['nmse']:.4f} "
                  f"cos={probe['cosine']:.4f}, selected={best_metrics['cosine']:.4f} @{best_step}")
    model.load_state_dict(best_state, strict=True)
    model.eval()
    return dict(phase=phase, steps=steps, updates=updates, attempts=attempts,
                best_step=best_step, probe=best_metrics, history=history)


In [ ]:
#@title 7. E2E teacher agreement and GPU latency of actual modules
@torch.no_grad()
def validate_pair(student, items, limit=None):
    teacher.eval()
    student.eval()
    limit = len(items) if limit is None else min(int(limit), len(items))
    n = matched = teacher_gold = student_gold = 0
    jensen = []
    for offset in range(0, limit, BATCH):
        chosen = items[offset:min(offset+BATCH, limit)]
        b = direct_batch(chosen)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b)
            sl, _ = student(**b)
        require_finite(tl, "teacher logits")
        require_finite(sl, "student logits")
        mask = b["marker_mask"].bool()
        tp = torch.softmax(tl.float().masked_fill(~mask, -1e9), dim=-1)
        sp = torch.softmax(sl.float().masked_fill(~mask, -1e9), dim=-1)
        gold = torch.tensor([c["gold_index"] for c in chosen], device=device)
        n += len(chosen)
        matched += int((tp.argmax(-1) == sp.argmax(-1)).sum())
        teacher_gold += int((tp.argmax(-1) == gold).sum())
        student_gold += int((sp.argmax(-1) == gold).sum())
        mid = .5*(tp+sp)
        j = .5*((tp*(tp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1))
        j += .5*((sp*(sp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1))
        jensen.extend(j.cpu().tolist())
    return dict(decisions=n, agreement=matched/max(n,1), mean_js=float(np.mean(jensen)),
                teacher_gold_accuracy=teacher_gold/max(n,1),
                student_gold_accuracy=student_gold/max(n,1),
                gold_accuracy_drop=(teacher_gold-student_gold)/max(n,1))

@torch.inference_mode()
def time_module(module, entry, warmup=5, repeats=20):
    module.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        for _ in range(warmup):
            module(entry["x"], position_embeddings=entry["pos"],
                   attention_mask=entry["mask"])
        torch.cuda.synchronize()
        times = []
        for _ in range(repeats):
            start = time.perf_counter()
            module(entry["x"], position_embeddings=entry["pos"],
                   attention_mask=entry["mask"])
            torch.cuda.synchronize()
            times.append((time.perf_counter()-start)*1000)
    return float(np.median(times))

@torch.inference_mode()
def time_model(model, chosen, warmup=3, repeats=10):
    b = direct_batch(chosen)
    model.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        for _ in range(warmup): model(**b)
        torch.cuda.synchronize()
        times=[]
        for _ in range(repeats):
            start=time.perf_counter()
            model(**b)
            torch.cuda.synchronize()
            times.append((time.perf_counter()-start)*1000)
    return float(np.median(times))

def length_diagnostics(items):
    lengths=np.array([len(it["ids"]) for it in items], dtype=np.int64)
    return dict(items=len(items), median_tokens=float(np.median(lengths)),
                p90_tokens=float(np.percentile(lengths,90)),
                max_tokens=int(lengths.max()),
                count_ge_256=int((lengths >= 256).sum()),
                count_ge_512=int((lengths >= 512).sum()))

@torch.no_grad()
def evaluate_candidate(layer_idx, replacement, audit_cache, items=audit_items):
    original = student_eval.encoder.layers[layer_idx].attn
    student_eval.encoder.layers[layer_idx].attn = replacement
    try:
        replacement.eval()
        local=local_metrics_from_cache(replacement, audit_cache)
        downstream=validate_pair(student_eval, items)
        entry=audit_cache[0]
        teacher_ms=time_module(teacher.encoder.layers[layer_idx].attn, entry)
        student_ms=time_module(replacement, entry)
        fixed=items[:min(BATCH,len(items))]
        teacher_total=time_model(teacher, fixed)
        student_total=time_model(student_eval, fixed)
        return dict(local=local, e2e=downstream, attention_ms=student_ms,
                    teacher_attention_ms=teacher_ms,
                    attention_speedup=teacher_ms/max(student_ms,1e-8),
                    model_ms=student_total, teacher_model_ms=teacher_total,
                    model_speedup=teacher_total/max(student_total,1e-8))
    finally:
        student_eval.encoder.layers[layer_idx].attn = original
        student_eval.eval().requires_grad_(False)

def gate(e):
    local=e["local"]
    downstream=e["e2e"]
    return bool(local["cosine"] >= .95 and local["nmse"] <= .10
                and downstream["agreement"] >= .95
                and downstream["mean_js"] <= .02)

print("Probe length profile:", length_diagnostics(probe_items))
print("Audit length profile:", length_diagnostics(audit_items))
print("Teacher baseline:", validate_pair(student_eval, audit_items))


In [ ]:
#@title 8. Screen four MultiscaleCeNN candidates at layer 15 (400 updates each)
train_cache = cache_teacher_io(LAYER, train_items, TRAIN_CACHE_BATCHES, offset=13*LAYER)
probe_cache = cache_teacher_io(LAYER, probe_items, PROBE_CACHE_BATCHES, offset=0)
audit_cache = cache_teacher_io(LAYER, audit_items, AUDIT_CACHE_BATCHES, offset=0)
screen_results = []
for i, name in enumerate(CANDIDATE_NAMES):
    random.seed(SEED + i*101)
    np.random.seed(SEED + i*101)
    torch.manual_seed(SEED + i*101)
    torch.cuda.manual_seed_all(SEED + i*101)
    print("\n" + "="*90)
    print("V2.9 FULL-ATTENTION REPLACEMENT —", name.upper(), "LAYER", LAYER)
    print("="*90)
    replacement=build_candidate(name, teacher.encoder.layers[LAYER].attn)
    training=fit_local(name, replacement, train_cache, probe_cache, STEPS)
    final=evaluate_candidate(LAYER, replacement, audit_cache)
    path=OUTPUT_DIR / f"layer{LAYER}_{name}_screen.pt"
    torch.save(dict(version="2.9", name=name, layer=LAYER,
                    config=CANDIDATE_INFO[name], state_dict=state_cpu(replacement),
                    train=training, audit=final), path)
    rec=dict(name=name, train=training, audit=final, checkpoint=str(path),
             rank_score=metric_score(training["probe"]),
             strict_gate=gate(final))
    screen_results.append(rec)
    print(f"RESULT {name}: probe_cos={training['probe']['cosine']:.4f} "
          f"audit_cos={final['local']['cosine']:.4f} "
          f"audit_agreement={final['e2e']['agreement']:.4f} "
          f"attention_speedup={final['attention_speedup']:.3f}x "
          f"model_speedup={final['model_speedup']:.3f}x")
    del replacement
    torch.cuda.empty_cache()

# Select from PROBE only: the audit set must not drive model selection.
screen_results.sort(key=lambda r: (r["rank_score"], -r["train"]["probe"]["cosine"]))
print("\nSCREEN ORDER (probe-selected):")
for idx, r in enumerate(screen_results, 1):
    print(f"{idx}. {r['name']:<17} cosine={r['train']['probe']['cosine']:.4f} "
          f"NMSE={r['train']['probe']['nmse']:.4f}")


In [ ]:
#@title 9. Optional longer refinement of top probe-selected architectures + final audit
refined_results=[]
if REFINE_STEPS > 0:
    for rank, r in enumerate(screen_results[:REFINE_TOP_K], 1):
        name=r["name"]
        print(f"\nREFINE {rank}/{min(REFINE_TOP_K,len(screen_results))}: {name} for {REFINE_STEPS} steps")
        replacement=build_candidate(name, teacher.encoder.layers[LAYER].attn)
        checkpoint=torch.load(r["checkpoint"], map_location="cpu", weights_only=False)
        replacement.load_state_dict(checkpoint["state_dict"], strict=True)
        refinement=fit_local(name, replacement, train_cache, probe_cache, REFINE_STEPS, phase="refine")
        evaluation=evaluate_candidate(LAYER, replacement, audit_cache)
        path=OUTPUT_DIR / f"layer{LAYER}_{name}_refined.pt"
        torch.save(dict(version="2.9", name=name, layer=LAYER,
                        config=CANDIDATE_INFO[name], state_dict=state_cpu(replacement),
                        screen=r["train"], refine=refinement, audit=evaluation), path)
        refined_results.append(dict(name=name, screen=r["train"], refinement=refinement,
                                    audit=evaluation, checkpoint=str(path),
                                    rank_score=metric_score(refinement["probe"]),
                                    strict_gate=gate(evaluation)))
        del replacement
        torch.cuda.empty_cache()

# Define winner from probe metric alone, not held-out audit.
pool = refined_results if refined_results else screen_results
ranked=sorted(pool, key=lambda r: (r["rank_score"], -(
    r["refinement"]["probe"]["cosine"] if "refinement" in r
    else r["train"]["probe"]["cosine"]
)))
winner=ranked[0]
print("\nFINAL PROBE-SELECTED ARCHITECTURE:", winner["name"])
print("Selected checkpoint:", winner["checkpoint"])
print("Held-out audit metrics:", json.dumps(winner["audit"], indent=2))
print("Passes predefined fidelity/decision gates:", winner["strict_gate"])

report = {
    "version": "2.9", "source_model": SOURCE_MODEL, "layer": LAYER,
    "attention_free_replacements": True, "teacher_has_full_attention": True,
    "official_test_used": False, "train_cases": TRAIN_CASES,
    "probe_cases": len(probe_rows), "audit_cases": len(audit_rows),
    "screen_steps": STEPS, "refine_steps": REFINE_STEPS,
    "winner_by_probe": winner["name"], "selected_checkpoint": winner["checkpoint"],
    "pass_quality_gate": winner["strict_gate"],
    "audit_lengths": length_diagnostics(audit_items),
    "screen_results": screen_results, "refined_results": refined_results,
    "selection_rule": "smallest probe NMSE + 1.10*(1-cosine), no audit-informed selection"
}
report_path=OUTPUT_DIR/"v29_multiscale_cenn_report.json"
report_path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
print("Saved:", report_path)

import matplotlib.pyplot as plt
for r in screen_results:
    h=r["train"]["history"]
    plt.plot([a["step"] for a in h], [a["cosine"] for a in h],
             marker=".", label=r["name"])
plt.axhline(.95, linestyle="--", color="gray", label="0.95 objective")
plt.title("Layer 15: attention-free CeNN output fidelity at 400 steps")
plt.xlabel("Successful optimizer updates")
plt.ylabel("Probe cosine")
plt.grid(alpha=.3)
plt.legend()
plt.show()


In [ ]:
#@title 10. Optional: replace ALL full-attention layers and validate cumulative drift
RUN_ALL_FULL_LAYERS = False #@param {type:"boolean"}
ALL_LAYERS_STEPS = 400 #@param {type:"integer"}
if RUN_ALL_FULL_LAYERS:
    # Train every full-attention layer independently against the untouched teacher.
    # Sliding-attention blocks are NEVER replaced in this optional experiment.
    installed=[]
    restored={}
    winner_name=winner["name"]
    try:
        for layer_idx in full_layers:
            old=student_eval.encoder.layers[layer_idx].attn
            restored[layer_idx]=old
            candidate=build_candidate(winner_name, teacher.encoder.layers[layer_idx].attn)
            if layer_idx == LAYER:
                saved=torch.load(winner["checkpoint"], map_location="cpu", weights_only=False)
                candidate.load_state_dict(saved["state_dict"], strict=True)
            else:
                layer_train=cache_teacher_io(layer_idx, train_items, TRAIN_CACHE_BATCHES,
                                             offset=13*layer_idx)
                layer_probe=cache_teacher_io(layer_idx, probe_items, PROBE_CACHE_BATCHES,
                                             offset=0)
                fit_local(winner_name, candidate, layer_train, layer_probe,
                          ALL_LAYERS_STEPS, "screen")
                del layer_train, layer_probe
            candidate.eval().requires_grad_(False)
            student_eval.encoder.layers[layer_idx].attn = candidate
            score=validate_pair(student_eval, audit_items)
            installed.append(dict(layer=layer_idx, agreement=score["agreement"],
                                  mean_js=score["mean_js"],
                                  gold_accuracy_drop=score["gold_accuracy_drop"]))
            print(f"After replacing full-attention layer {layer_idx}: "
                  f"agreement={score['agreement']:.4f} JS={score['mean_js']:.6f}")
        all_path=OUTPUT_DIR/"v29_all_full_layers_cumulative.json"
        all_path.write_text(json.dumps(installed, indent=2), encoding="utf-8")
        print("Cumulative report:", all_path)
    finally:
        for idx, old in restored.items():
            student_eval.encoder.layers[idx].attn = old
        student_eval.eval().requires_grad_(False)
        torch.cuda.empty_cache()
else:
    print("All-full-layers experiment is disabled. Enable only after isolated layer 15 tests.")
